[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-02-ML-Classification <<](./QC-Py-Cloud-02-ML-Classification.ipynb) | [Suivant : QC-Py-Cloud-03-DualMomentum >>](./QC-Py-Cloud-03-DualMomentum.ipynb)



# QC-Py-Cloud-02 — Sector Rotation & Multi-Asset Momentum



**Module**: Hands-On AI Trading, Ch.7 — Sector Rotation & Momentum Strategies



**Objectif**: Tester et comparer des approches de rotation sectorielle et de trend-following multi-actifs sur QuantConnect Cloud. Partir d'une strategie sector rotation classique et evoluer vers un trend-following diversifie inspire d'AQR.



**Approche cloud-native**: L'algorithme est execute sur QuantConnect Cloud. Les resultats sont syncs ci-dessous.


> **Note de conception** : Ce notebook est un descriptif pedagogique. Le code executable se trouve dans le projet QuantConnect Cloud correspondant (voir le lien dans le notebook). L'absence d'outputs est intentionnelle.


## 1. Concept : Sector Rotation vs Multi-Asset Trend Following

### Sector Rotation
La rotation sectorielle consiste a allouer le capital vers les secteurs les plus performants du moment, en se basant sur le momentum relatif. L'idee est que les secteurs en tendance outperform les secteurs en declin sur des horizons de 3 a 12 mois.

**Hypothese** : Le momentum sectoriel persiste (les gagnants continuent de gagner a court terme).

### Multi-Asset Trend Following
Approche inspiree des hedge funds trend-following (AQR, Man AHL). Au lieu de selectionner parmi des secteurs correles (meme marche actions US), on investit dans des classes d'actifs decorrelees en suivant la tendance.

**Principe AQR** (Hurst, Ooi, Pedersen, 2014) :
- Filtre double : prix > SMA200 ET momentum 6 mois > 0
- Allocation egale parmi les actifs qualifies
- Si aucun actif ne qualifie : position defensive (T-bills)

**Avantage** : Diversification跨 asset classes, pas de correlation sectorielle.

## 2. Univers d'actifs

### v1-v2 : Sector ETFs (11 secteurs GICS)

| Ticker | Secteur | Correlation SPY |
|--------|---------|----------------|
| XLK | Technology | Haute |
| XLY | Consumer Disc. | Haute |
| XLF | Financials | Haute |
| XLE | Energy | Moyenne |
| XLV | Healthcare | Moyenne |
| XLI | Industrials | Haute |
| XLP | Consumer Staples | Moyenne |
| XLB | Materials | Haute |
| XLU | Utilities | Basse |
| XLRE | Real Estate | Moyenne |
| IBB | Biotech | Moyenne |

**Probleme** : La plupart des secteurs sont fortement correles entre eux. La rotation sectorielle ne genere pas de veritable diversification.

### v3-v4 : Diversified Asset ETFs

| Ticker | Classe d'actif | Role |
|--------|---------------|------|
| QQQ | Nasdaq-100 | Growth, tech leverage |
| SPY | S&P 500 | Large-cap equity |
| EFA | International (EAFE) | Diversification geographique |
| GLD | Or | Inflation hedge, crisis alpha |
| IWM | Russell 2000 | Small-cap equity |
| SHY | Treasury 1-3y (defensif) | Cash proxy quand tendance negative |

## 3. Methodologie

Quatre versions ont ete testees sur QC Cloud (projet 30821748) :

### v1 — Sector Rotation (11 secteurs, top 3, momentum 6m)
- Univers : 11 sector ETFs (XLK, XLY, XLF, XLE, XLV, XLI, XLP, XLB, XLU, XLRE, IBB)
- Signal : Momentum 6 mois (252 trading days / 2)
- Selection : Top 3 secteurs par momentum
- Allocation : Equal-weight (33% chacun)
- Rebalance : Mensuel (21 jours)

### v2 — Sector Rotation concentre (8 secteurs, top 2, momentum 3m)
- Univers : 8 sector ETFs (retire XLU, XLRE, IBB)
- Signal : Momentum 3 mois (63 trading days)
- Selection : Top 2 secteurs
- Allocation : Equal-weight (50% chacun)
- Rebalance : Mensuel

### v3 — Multi-Asset Trend Following (AQR Hybrid)
- Univers : 5 ETFs diversifies (QQQ, SPY, EFA, GLD, IWM) + SHY (defensif)
- Signal : Prix > SMA200 ET momentum 6 mois > 0 (double filtre)
- Allocation : Equal-weight parmi actifs qualifies
- Si 0 actif qualifie : 100% SHY (defensif)
- Rebalance : Mensuel (21 jours)

### v4 — Momentum-Weighted Trend Following
- Meme univers et filtres que v3
- Allocation : Proportionnelle au score de momentum (momentum-weighted)
- Objectif : Surponderer les actifs avec le momentum le plus fort
- Rebalance : Mensuel

## 4. Resultats QC Cloud

**Projet QC Cloud**: 30821748 (`Cloud-SectorRotation-Momentum`)
**Periode**: 2014-01-01 au 2025-01-01 (11 ans)
**Capital initial**: $100,000

| Version | Univers | Allocation | Sharpe | CAGR | MaxDD | Net Profit | Ordres |
|---------|---------|------------|--------|------|-------|------------|--------|
| v1 (11 secteurs, top 3) | Sector ETFs | Equal-weight | 0.233 | 5.81% | 21.6% | +86.1% | 468 |
| v2 (8 secteurs, top 2) | Sector ETFs | Equal-weight | 0.240 | 5.54% | 15.7% | +81.0% | 517 |
| **v3 (5 ETFs, AQR)** | **Multi-asset** | **Equal-weight** | **0.614** | **10.76%** | **15.5%** | **+207.8%** | **474** |
| v4 (5 ETFs, mom-wt) | Multi-asset | Mom-weighted | 0.276 | 6.67% | 38.8% | +103.5% | 557 |

### Benchmark : SPY Buy & Hold
Sur la meme periode, SPY a un CAGR ~12.8% et un MaxDD ~24%. Le v3 ne bat pas SPY en rendement pur, mais offre un meilleur ratio rendement/risque (Sharpe 0.614 vs ~0.55 pour SPY).

### Exercice 1 : Double filtre tendance AQR

L'approche AQR utilise un double filtre : prix > SMA200 ET momentum 6 mois > 0. Ce filtre elimine les actifs en bear market. La v3 atteint un Sharpe de 0.614 grace a ce filtre.

**Objectif** : Implementez `aqr_filter(prices)` qui retourne la liste des actifs eligibles selon le double filtre. Testez avec des series simulees en bull, bear et sideways.

**Indices** :
- # Indice : Un actif en bull aura prix > SMA200 ET momentum_6m > 0.
- # Indice : En bear market, la plupart des actifs seront exclus par le filtre.

In [ ]:
# Exercice 2 : Double filtre tendance AQR
# TODO etudiant : implementer aqr_filter(prices_dict) -> list des tickers eligibles
# Indice : eligible si prix[-1] > SMA(200) ET momentum_6m > 0

def aqr_filter(prices_dict, sma_period=200, mom_period=126):
    # TODO etudiant : pour chaque ticker, verifier les deux conditions
    return None  # TODO etudiant : retourner la liste des eligibles

result_aqr = None  # TODO etudiant : tester avec des series simulees
print("Exercice a completer : double filtre AQR")

## 5. Analyse : Pourquoi le Multi-Asset bat le Sector Rotation

### v1-v2 : Sector Rotation sous-performe

La rotation sectorielle (v1, v2) ne genere pas d'alpha significatif (Sharpe ~0.23-0.24). Les raisons :

1. **Correlation elevee entre secteurs** : Les 11 secteurs GICS sont tous correlates au marche US global. Quand le S&P 500 chute, presque tous les secteurs chutent ensemble. La rotation ne protege pas.

2. **Momentum sectoriel est du beta deplace** : Le momentum d'un secteur est largement explique par le beta du secteur au marche.XLK a un beta > 1 donc son momentum "outperform" en bull market mais chute plus en bear.

3. **Concentration augmente le risque** : Selectionner les top 2-3 secteurs concentre le portefeuille sur 2-3 positions, augmentant la volatilite sans compensation adequate.

### v3 : Multi-Asset Trend Following excelle

Le passage a un univers diversifie (v3) triple le Sharpe (0.233 -> 0.614). Pourquoi :

1. **Decorrelation veritable** : QQQ (tech), GLD (or), EFA (international), IWM (small caps) ont des cycles economiques differents. Quand tech chute, l'or peut monter.

2. **Filtre tendance = protection drawdown** : Le double filtre (SMA200 + momentum 6m > 0) elimine les actifs en bear market. En 2022, SPY et QQQ etaient exclus, GLD etait inclus.

3. **Position defensive (SHY)** : Quand aucun actif ne qualifie, le portefeuille se refugie en T-bills. Cela protege le capital pendant les crises globales.

### v4 : Momentum-weighting echoue

La ponderation par momentum (v4) degrade la performance (Sharpe 0.276, MaxDD 38.8%). Pourquoi :

1. **Concentration dynamique** : Le momentum-weighting surpondere l'actif le plus "chaud". En 2020-2021, QQQ avait le momentum le plus fort et recevait 40-50% du capital. Quand QQQ a corrige en 2022, le drawdown a ete brutal.

2. **Retournement de momentum** : Les actifs avec le momentum le plus fort sont aussi ceux qui ont le plus a perdre quand la tendance s'inverse. Le momentum-weighting maximise l'exposition au retournement.

3. **Lecon : L'equal-weight est plus robuste** : Sans signal de confiance sur la persistance du momentum par actif, l'equal-weight est plus prudent et plus stable.

### Exercice 2 : Comparaison equal-weight vs momentum-weighted

La v4 (momentum-weighted) sous-performe la v3 (equal-weight) avec un Sharpe de 0.276 vs 0.614 et un MaxDD de 38.8% vs 15.5%. La ponderation par momentum concentre trop le portefeuille sur l'actif le plus chaud.

**Objectif** : Simulez les poids equal-weight et momentum-weighted pour 4 actifs avec des scores de momentum differents. Calculez la concentration (indice Herfindahl-Hirschman, somme des carres des poids) pour chaque methode. Un HHI plus eleve indique une plus grande concentration.

**Indices** :
- # Indice : HHI = sum(w_i^2). Pour equal-weight avec N actifs, HHI = 1/N.
- # Indice : Momentum-weighting normalise les scores de momentum pour sommer a 1.

In [ ]:
# Exercice 3 : Comparaison equal-weight vs momentum-weighted
# TODO etudiant : calculer les poids equal-weight et momentum-weighted
# TODO etudiant : calculer la concentration HHI pour chaque methode
# Indice : HHI = sum(w_i^2), plus HHI est haut, plus c est concentre

def hhi(weights):
    # TODO etudiant : calculer l indice Herfindahl-Hirschman
    return None  # TODO etudiant : retourner la somme des carres

mom_scores = {"QQQ": 0.15, "SPY": 0.08, "EFA": -0.02, "GLD": 0.05}
result_alloc = None  # TODO etudiant : comparer HHI(equal) vs HHI(momentum-weighted)
print("Exercice a completer : equal-weight vs momentum-weighted")

### Exercice 3 : Calcul du momentum relatif

La rotation sectorielle selectionne les secteurs avec le meilleur momentum relatif. Le momentum 6 mois est calcule comme le rendement sur 126 jours de bourse.

**Objectif** : Implementez une fonction `relative_momentum(prices, lookback=126)` qui prend un DataFrame de prix (lignes = dates, colonnes = tickers) et retourne le classement par momentum. Identifiez le top 3.

**Indices** :
- # Indice : Le momentum 6m = prix_actuel / prix_actuel_minus_126 - 1.
- # Indice : Triez par momentum decroissant et prenez les 3 premiers.

In [ ]:
# Exercice 1 : Calcul du momentum relatif
# TODO etudiant : implementer relative_momentum(prices, lookback=126) -> classement
# Indice : momentum = prix[-1] / prix[-lookback] - 1

import numpy as np

def relative_momentum(prices_dict, lookback=126):
    # prices_dict : {ticker: array_de_prix}
    # TODO etudiant : calculer le momentum pour chaque ticker
    # TODO etudiant : trier par momentum decroissant
    return None  # TODO etudiant : retourner le classement

# Test avec des donnees simulees
np.random.seed(42)
sim_prices = {t: 100 + np.cumsum(np.random.randn(300) * (0.3 + 0.1*i)) for i, t in enumerate(["XLK","XLY","XLF","XLE","XLV"])}
result_mom = None  # TODO etudiant : appeler relative_momentum
print("Exercice a completer : momentum relatif sectoriel")


## 6. Code source (v3 — meilleur resultat)

Le code est disponible sur QC Cloud (projet 30821748). Le notebook local ne contient que les resultats et l'analyse, conformement au workflow cloud-native.

```python
# Lien QC Cloud : https://www.quantconnect.com/project/30821748
# Approche : SMA200 + 6m momentum dual filter, equal-weight, SHY defensive
# Rebalance mensuel, 5 ETFs diversifies
#
# Points cles du code :
# 1. Double filtre : prix > SMA(200) AND ROC(126) > 0
# 2. Equal-weight parmi les actifs qualifies
# 3. Position defensive SHY si aucun actif ne qualifie
# 4. Rebalance toutes les 21 jours de bourse
```

## 7. Comparaison avec le Risk Parity (Cloud-01)

| Strategie | Sharpe | CAGR | MaxDD | Net Profit |
|-----------|--------|------|-------|------------|
| Risk Parity v4 (Cloud-01) | 0.278 | 6.17% | 20.4% | +93.3% |
| **Sector Rotation v3 (Cloud-02)** | **0.614** | **10.76%** | **15.5%** | **+207.8%** |

Le trend-following multi-asset surpasse le Risk Parity sur tous les metrics. La principale difference : le trend-following filtre activement les actifs en tendance negative, tandis que le Risk Parity reste expose a tous les actifs en permanence (avec une ponderation differente).

**Lecon** : Un filtre de tendance actif (SMA200 + momentum) est plus efficace qu'une allocation statique basee sur la volatilite.

## 8. Pour aller plus loin

1. **Volatility targeting** : Ajuster l'exposition globale pour cibler un niveau de vol (ex: 10% annuel). Reduirait le drawdown.

2. **Speed exit** : Ajouter un stop-loss rapide quand un actif passe sous son SMA50 (pas seulement au rebalance). Limite les pertes intra-mois.

3. **Regime filter** : Combinaison avec un detecteur de regime (bull/bear/sideways via VIX ou moving averages) pour ajuster l'aggressivite de l'allocation.

4. **Universe etendu** : Ajouter des ETFs internationaux supplmentaires (EEM, DBC) et des ETFs alternatifs (TLT quand les taux se stabiliseront).

**Reference** : Hurst, Ooi, Pedersen (2014) — "A Century of Evidence on Trend-Following Investing", AQR. Moskowitz, Ooi, Pedersen (2012) — "Time Series Momentum", Journal of Financial Economics.